# Modul A · Kapitel 2 · Teil 1 — Vom Text zum Vektor

## Challenge: Tokenizer, Word Embedding, Positional Encoding

**Lernziel:** Du kannst erklären, wie aus einem Text die Zahlen werden, mit denen ein
Sprachmodell rechnet. Dafür baust du die drei Schritte selbst:

1. Tokenisierung,
2. Token-Embedding,
3. und Positional Encoding.

Das ist der Eingang jedes Decoder-Only-Modells:

```
Eingabe
   │
   ▼
Tokenizer ────────────► Word Embedding ──┐
   │                                     ├──► …
   └──────────────────► Positional       │
                        Encoding ────────┘
```

### So funktioniert dieses Notebook

| Symbol | Bedeutung |
|:--:|---|
| 📖 | Erklärung — lesen |
| ▶️ | Fertiger Code — einfach ausführen (`Shift` + `Enter`) |
| 🛠️ | **Challenge** — hier schreibst du selbst Code |
| ✅ | Selbsttest — sagt dir sofort, ob deine Lösung stimmt |
| 💡 | **Lösung** — zum Aufklappen, wenn du nicht weiterkommst |
| 💬 | Diskussionsfrage — kurz überlegen, gerne mit der Nachbarin / dem Nachbarn |

**Wichtig:** Führe die Zellen **von oben nach unten** aus. Spätere Zellen brauchen die Funktionen, die du vorher schreibst.

Es sind insgesamt **4 Challenges**.

---
## 0 · Setup

▶️ Führe diese zwei Zellen aus.

Neu ist **PyTorch**. Wir benutzen es hier nur als Rechenbibliothek für Vektoren und Matrizen —
das Training kommt später. In Colab ist PyTorch bereits installiert, lokal: `pip install torch`.

Die zweite Zelle besorgt den Text, mit dem wir arbeiten: **Faust I und II** von Goethe,
gemeinfrei, rund eine halbe Megabyte.

In [ ]:
import re
from collections import Counter
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.nn import functional as F

# Einheitliche Farben für alle Diagramme in diesem Notebook
BLAU, ORANGE, TEAL, GRAU = "#2563eb", "#e8590c", "#0d9488", "#6b7280"

plt.rcParams.update({
    "figure.figsize": (8, 5),
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": GRAU,
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": "#e5e7eb",
    "grid.linewidth": 0.8,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "font.size": 11,
})
torch.manual_seed(1337)

print(f"PyTorch {torch.__version__}")
print("Setup fertig ✔")

In [ ]:
def lade_faust():
    """Lädt Goethes Faust I und II — einmalig von Project Gutenberg, danach aus `faust.txt`."""
    import urllib.request

    pfad = Path("faust.txt")
    if pfad.exists():
        print(f"Gelesen: {pfad}")
        return pfad.read_text(encoding="utf-8")

    print("faust.txt nicht gefunden — lade von Project Gutenberg (rund 0,5 MB) …")

    def teil(nummer, ab):
        """Holt ein Buch und schneidet Vorspann, Nachspann und Inhaltsverzeichnis weg."""
        adresse = f"https://www.gutenberg.org/cache/epub/{nummer}/pg{nummer}.txt"
        roh = urllib.request.urlopen(adresse).read().decode("utf-8").replace("\r\n", "\n")
        roh = roh[:roh.find("*** END OF THE PROJECT GUTENBERG")]
        roh = "\n".join(z[2:] if z.startswith("  ") else z for z in roh.split("\n"))
        return roh[roh.find(ab):]

    erster = teil(2229, "Zueignung\n\n\nIhr naht")
    zweiter = teil(2230, "1.  Akt--Anmutige Gegend")
    # Faust II schreibt die Sprechernamen mit Doppelpunkt, Faust I mit Punkt — wir vereinheitlichen
    zweiter = re.sub(r"(?m)^([A-ZÄÖÜ][A-ZÄÖÜ \-]*(?:\(.*?\))?):$", r"\1.", zweiter)

    text = re.sub(r"\n{3,}", "\n\n", erster.strip() + "\n\n" + zweiter.strip()) + "\n"
    Path("faust.txt").write_text(text, encoding="utf-8")
    print("Gespeichert als: faust.txt")
    return text


text = lade_faust()

print(f"{len(text):,} Zeichen".replace(",", "."))

---
## 1 · Tokenisierung: aus Text werden Zahlen

📖 Ein neuronales Netz rechnet mit Zahlen, nicht mit Buchstaben. Der erste Schritt jedes
Sprachmodells ist deshalb die **Tokenisierung**: Text in eine Liste von Zahlen zerlegen.

Die Frage ist nur: Was ist ein **Token**?

| Ansatz | „Guten Morgen" wird zu | Vokabular | Nachteil |
|---|---|---|---|
| **Zeichen** | `G, u, t, e, n, ␣, M, …` | 87 | lange Folgen, wenig Bedeutung je Token |
| **Wortteile** (BPE) | `G`, `uten`, `␣Morg`, `en` | 50.000 – 200.000 | muss vorher gelernt werden |
| **Wörter** | `Guten`, `Morgen` | Millionen | unbekannte Wörter unmöglich |

Wir fangen mit **Zeichen** an: Das Vokabular ist winzig, und der Tokenizer passt in zwei Zeilen.
Danach bauen wir die Wortteile selbst und vergleichen mit dem Tokenizer von GPT-2.

▶️ Schauen wir zuerst in den Text:

In [ ]:
print(text[:100])
print("…")

In [ ]:
chars = sorted(set(text))
vokabular_groesse = len(chars)

print(f"Der Text besteht aus {vokabular_groesse} verschiedenen Zeichen:")
print("".join(chars))
print()
print(f"Länge des gesamten Textes: {len(text):,} Zeichen".replace(",", "."))

📖 87 verschiedene Zeichen — Buchstaben, Umlaute, Satzzeichen, Zeilenumbruch. Das ist das
komplette Vokabular, wenn ein Token ein Zeichen ist.

Wir brauchen zwei Übersetzungstabellen: von Zeichen zu Zahl und wieder zurück.

In [ ]:
# ▶️ Die beiden Tabellen: Zeichen → Zahl und Zahl → Zeichen
zeichen_zu_zahl = {z: i for i, z in enumerate(chars)}
zahl_zu_zeichen = {i: z for i, z in enumerate(chars)}

for z in ["F", "a", "u", "s", "t", " ", "ö"]:
    print(f"  {z!r:>5}  →  {zeichen_zu_zahl[z]:>3}")

### 🛠️ Challenge 1 — Der Tokenizer

Schreibe die beiden Funktionen:

* `kodiere(s)` nimmt einen Text und gibt eine **Liste von Zahlen** zurück
* `dekodiere(zahlen)` nimmt eine Liste von Zahlen und gibt den **Text** zurück

*Tipp: Beides sind Einzeiler mit einer Listen-Abkürzung. `[zeichen_zu_zahl[z] for z in s]`
läuft durch alle Zeichen von `s` und schlägt jedes in der Tabelle nach. Zum Zusammensetzen von
Zeichen zu einem Text nimmt man `"".join(...)`.*

In [ ]:
def kodiere(s):
    """Text → Liste von Zahlen."""
    # TODO: Ersetze die nächste Zeile
    raise NotImplementedError("Challenge 1: kodiere() implementieren")


def dekodiere(zahlen):
    """Liste von Zahlen → Text."""
    # TODO: Ersetze die nächste Zeile
    raise NotImplementedError("Challenge 1: dekodiere() implementieren")

In [ ]:
# ✅ Selbsttest
assert kodiere("Faust") == [zeichen_zu_zahl[z] for z in "Faust"]
assert len(kodiere("Habe nun, ach!")) == 14, "Ein Token je Zeichen — Leerzeichen zählen mit"
assert dekodiere(kodiere("Habe nun, ach! Philosophie")) == "Habe nun, ach! Philosophie"
assert dekodiere([]) == "", "Leere Liste ergibt leeren Text"
print("✅ Challenge 1 gelöst")
print()
print("Habe nun, ach!  →  ", kodiere("Habe nun, ach!"))
print("und wieder zurück:  ", repr(dekodiere(kodiere("Habe nun, ach!"))))

<details>
<summary>💡 Lösung aufklappen — erst selbst probieren!</summary>

```python
def kodiere(s):
    """Text → Liste von Zahlen."""
    return [zeichen_zu_zahl[z] for z in s]


def dekodiere(zahlen):
    """Liste von Zahlen → Text."""
    return "".join(zahl_zu_zeichen[i] for i in zahlen)
```

</details>

📖 Das ist ein vollständiger Tokenizer. Er macht aus jedem Zeichen ein Token, und damit werden die Folgen lang.

In [ ]:
# ▶️ Der komplette Faust als Zahlenfolge
daten = torch.tensor(kodiere(text), dtype=torch.long)

print(f"Form:  {tuple(daten.shape)}   {daten.dtype}")
print()
print("Die ersten 60 Zeichen als Zahlen:")
print(daten[:60].tolist())
print()
print("… und wieder zurück:")
print(repr(dekodiere(daten[:60].tolist())))

---
### 1.2 · Zum Vergleich: der Tokenizer von GPT-2

📖 Echte Modelle zerlegen Text nicht in Zeichen, sondern in **Wortteile**. Das Verfahren heißt
**Byte Pair Encoding** (BPE): Häufige Zeichenfolgen werden zu einem einzigen Token
zusammengefasst. Häufige Wörter bleiben ganz, seltene zerfallen in Stücke.

Der Tokenizer von GPT-2 hat **50.257** Tokens. Wir laden ihn und schicken denselben Satz durch
beide Tokenizer.

In [ ]:
# ▶️ Der Original-Tokenizer von OpenAI (kleiner Download beim ersten Aufruf)
GPT2 = None
try:
    try:
        import tiktoken
    except ImportError:
        %pip install -q tiktoken
        import tiktoken
    GPT2 = tiktoken.get_encoding("gpt2")
    print(f"GPT-2-Tokenizer geladen: {GPT2.n_vocab:,} Tokens".replace(",", "."))
except Exception as fehler:
    print("GPT-2-Tokenizer nicht verfügbar:", fehler)
    print("Nur dieser Vergleich entfällt dann — alles andere im Notebook läuft weiter.")


def gpt2_stuecke(s):
    """Zerlegt einen Text mit dem GPT-2-Tokenizer und gibt die Stücke als Text zurück."""
    return [GPT2.decode([nummer]) for nummer in GPT2.encode(s)]

In [ ]:
# ▶️ Derselbe Satz, zwei Tokenizer
satz = "Habe nun, ach! Philosophie"

print("Zeichen-Tokenizer:")
print(f"  {len(kodiere(satz))} Tokens: {[z for z in satz]}")
print()
if GPT2 is not None:
    print("GPT-2-Tokenizer:")
    print(f"  {len(GPT2.encode(satz))} Tokens: {gpt2_stuecke(satz)}")

In [ ]:
# ▶️ Derselbe Satz auf Deutsch und auf Englisch
if GPT2 is not None:
    for s in ["The quick brown fox jumps over the lazy dog.",
              "Der schnelle braune Fuchs springt über den faulen Hund."]:
        stuecke = gpt2_stuecke(s)
        print(f"{len(stuecke):>3} Tokens  {stuecke}")
        print()

    for wort in ["strawberry", "Erdbeere", "Donaudampfschiff"]:
        print(f"  {wort:<18} → {gpt2_stuecke(wort)}")

📖 Zwei Dinge, die man im Alltag ständig wiedersieht:

* **Deutsch kostet mehr.** Derselbe Satz braucht auf Deutsch etwa doppelt so viele Tokens wie
  auf Englisch — GPT-2 wurde überwiegend auf englischem Text trainiert, deutsche
  Umlaute zum Beispiel zerfallen in Bruchstücke, wodurch mehr Tokens gebraucht werden. Abgerechnet wird nach Tokens. Deutsch ist also teurer als Englisch.
* **Das Modell sieht keine Buchstaben.** `Erdbeere` besteht für GPT-2 aus vier Stücken. Die
  Frage „wie viele *r* stecken in Erdbeere" ist damit keine Frage über Buchstaben mehr, sondern
  eine über Tokens, die das Model kaum sicher beantworten kann.

Beim `ü` in *über* sieht man außerdem: GPT-2 arbeitet auf **Bytes**. Ein Umlaut besteht aus zwei
Bytes und wird zu zwei Tokens, die einzeln keinen darstellbaren Text ergeben — deshalb das `�`.

---
### 1.3 · Wortteile selbst bauen

📖 Wie kommt ein Tokenizer an seine Wortteile? Byte Pair Encoding ist ein sehr kurzes Rezept,
das nur zwei Schritte wiederholt:

1. Suche das **häufigste benachbarte Paar** von Tokens im Text.
2. Ersetze dieses Paar überall durch **ein neues Token** und schreibe es ins Vokabular.

Schritt 1 ist deine Challenge, Schritt 2 ist unten schon fertig.

### 🛠️ Challenge 2 — Das häufigste Paar finden

`haeufigstes_paar(ids)` bekommt eine Liste von Token-Nummern und gibt das **häufigste
benachbarte Paar** als Tupel zurück. Aus `[1, 2, 1, 2, 3]` wird `(1, 2)` — dieses Paar kommt
zweimal vor, alle anderen einmal.

*Tipp: `zip(ids, ids[1:])` läuft durch alle Nachbarpaare. `Counter(...)` zählt sie, und
`.most_common(1)` gibt das häufigste zurück — als Liste mit einem Eintrag `[(paar, anzahl)]`.*

In [ ]:
def haeufigstes_paar(ids):
    """Liste von Token-Nummern → das häufigste benachbarte Paar, z. B. (12, 45)."""
    # TODO: Ersetze die nächste Zeile
    raise NotImplementedError("Challenge 2: haeufigstes_paar() implementieren")

In [ ]:
# ✅ Selbsttest
assert haeufigstes_paar([1, 2, 1, 2, 3]) == (1, 2)
assert haeufigstes_paar([5, 5, 5]) == (5, 5)
assert haeufigstes_paar([7, 8]) == (7, 8), "Auch bei nur einem Paar muss ein Tupel kommen"
paar = haeufigstes_paar(kodiere(text[:50_000]))
assert isinstance(paar, tuple) and len(paar) == 2
print("✅ Challenge 2 gelöst")
print()
print(f"Häufigstes Paar in den ersten 50.000 Zeichen: {paar}"
      f"  =  {dekodiere(list(paar))!r}")

<details>
<summary>💡 Lösung aufklappen — erst selbst probieren!</summary>

```python
def haeufigstes_paar(ids):
    """Liste von Token-Nummern → das häufigste benachbarte Paar, z. B. (12, 45)."""
    return Counter(zip(ids, ids[1:])).most_common(1)[0][0]
```

</details>

▶️ Jetzt der zweite Schritt und die Wiederholung: 150-mal das häufigste Paar suchen und
verschmelzen. Das Vokabular wächst dabei von 87 auf 237, und die Zahlenfolge wird kürzer.

In [ ]:
# ▶️ Byte Pair Encoding auf einer Probe von 200.000 Zeichen (dauert ein paar Sekunden)
def verschmelze(ids, paar, neu):
    """Ersetzt jedes Vorkommen von `paar` durch die neue Token-Nummer `neu`."""
    ergebnis = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and (ids[i], ids[i + 1]) == paar:
            ergebnis.append(neu)
            i += 2
        else:
            ergebnis.append(ids[i])
            i += 1
    return ergebnis


VERSCHMELZUNGEN = 150

probe = text[:200_000]
ids = kodiere(probe)

stueck = dict(zahl_zu_zeichen)          # Token-Nummer → Textstück, wächst mit
neue_tokens = []                        # die gelernten Wortteile, der Reihe nach
laengen = [len(ids)]                    # Länge der Folge nach jeder Verschmelzung

for schritt in range(VERSCHMELZUNGEN):
    paar = haeufigstes_paar(ids)
    neue_nummer = vokabular_groesse + schritt
    ids = verschmelze(ids, paar, neue_nummer)
    stueck[neue_nummer] = stueck[paar[0]] + stueck[paar[1]]
    neue_tokens.append(stueck[neue_nummer])
    laengen.append(len(ids))

print(f"Vokabular:  {vokabular_groesse} → {vokabular_groesse + VERSCHMELZUNGEN} Tokens")
print(f"Folge:      {laengen[0]:,} → {laengen[-1]:,} Tokens".replace(",", "."))
print(f"Das sind {len(probe) / laengen[-1]:.2f} Zeichen je Token."
      .replace(".", ",", 1))
print()
print("Die ersten 40 gelernten Wortteile:")
print("  " + "  ".join(repr(t) for t in neue_tokens[:40]))

In [ ]:
# ▶️ Wie stark die Folge schrumpft — und wo GPT-2 auf demselben Text liegt
plt.figure(figsize=(8, 4.5))
plt.plot(range(len(laengen)), [len(probe) / n for n in laengen], color=BLAU, lw=2,
         label="selbst gebautes BPE")

if GPT2 is not None:
    gpt2_laenge = len(GPT2.encode(probe))
    plt.axhline(len(probe) / gpt2_laenge, color=ORANGE, lw=2, ls="--",
                label=f"GPT-2 (50.257 Tokens): {len(probe) / gpt2_laenge:.2f}")

plt.xlabel("Verschmelzungen")
plt.ylabel("Zeichen je Token")
plt.title("Größeres Vokabular → kürzere Folgen")
plt.legend()
plt.show()

📖 Nach 150 Verschmelzungen stehen im Vokabular genau die Bausteine, die man in deutschem Text
erwartet: `ch`, `en`, `er`, `sch`, dann ganze Wörter wie `und `, `der `, `die `, `nicht`. Das
Verfahren bekommt nirgends gesagt, was eine Silbe oder ein Wort ist — es zählt nur Häufigkeiten.

Der Vergleich mit GPT-2 fällt knapper aus als erwartet: Unsere 237 Tokens schaffen 1,86 Zeichen
je Token, die 50.257 Tokens von GPT-2 schaffen 2,25. Das 200-fache Vokabular bringt auf
deutschem Text also nur gut 15 % kürzere Folgen — weil das Vokabular von GPT-2 auf englischem
Text entstanden ist. Ein Tokenizer ist immer auf die Sprache zugeschnitten, aus der er gelernt
wurde.

**Für den Rest dieses Kapitels bleiben wir beim Zeichen-Tokenizer.** Er kostet uns längere
Folgen, spart aber das ganze BPE-Drumherum — und der Rest des Modells ist davon unberührt.

💬 **Warum arbeiten echte Sprachmodelle mit Wortteilen und nicht mit Zeichen — obwohl Zeichen
doch viel einfacher sind?**

<details>
<summary>Antwort aufklappen</summary>

Wegen der **Kontextlänge**. Ein Modell kann immer nur eine begrenzte Zahl von Tokens
gleichzeitig anschauen, und der Rechenaufwand der Aufmerksamkeit wächst **quadratisch** mit
dieser Zahl: doppelt so viele Tokens bedeuten viermal so viel Rechnung.

Mit Wortteilen braucht eine Seite Text rund 500 Tokens, mit Zeichen rund 2.000. Bei gleichem
Rechenbudget sieht das Modell mit Wortteilen also viermal so viel Text.

Dazu kommt: Ein Token wie `Morgen` trägt bereits Bedeutung. Ein einzelnes `M` trägt keine — das
Modell muss die Bedeutung erst aus mehreren Zeichen zusammensetzen, und das kostet Schichten.

</details>

---
## 2 · Word Embedding: aus Zahlen werden Vektoren

📖 Nach der Tokenisierung ist jedes Zeichen eine Zahl zwischen 0 und 86. Damit kann ein Netz
nicht rechnen: Die 40 ist nicht „doppelt so viel" wie die 20, und die 41 ist der 40 nicht
ähnlicher als die 12. Die Zahlen sind Namen, keine Größen.

Im Flussdiagramm heißt dieser Kasten **Word Embedding**. Weil unsere Tokens Zeichen sind und
keine Wörter, nennen wir die Tabelle hier **Token-Embedding** — gemeint ist dasselbe.

Der einfachste Ansatz ist **One-Hot-Encoding**:

Jedes Token wird in einen Vektor mit **87 Zahlen** umgewandelt – entsprechend der Größe unseres Vokabulars.

Der Vektor besteht aus:

* genau **einer 1**
* und **86 Nullen**

Angenommen, der Buchstabe **`c`** steht an dritter Stelle in unserem Vokabular. Dann sieht sein One-Hot-Vektor so aus:

`(0, 0, 1, 0, 0, ..., 0)`

Die Position der **1** zeigt also, um welches Token es sich handelt.

Nachteil ist es sind eben auch alle Tokens exakt gleich weit voneinander entfernt, d.h. das Modell kann nicht die Bedeutung eines Wortes oder Textes erlernen.

In [ ]:
# ▶️ One-Hot-Encoding: ein Vektor je Zeichen, so lang wie das Vokabular
probe_ids = torch.tensor(kodiere("Faust"))
one_hot = F.one_hot(probe_ids, num_classes=vokabular_groesse).float()

print(f"5 Zeichen  →  {tuple(one_hot.shape)}")
print()
print("Der Vektor für 'F' (Ausschnitt, Stelle 25 bis 45):")
print(one_hot[0, 25:45].numpy().astype(int))

📖 **Embeddings** geben jedem Token einen Zahlenvektor und damit eine mathematische Repräsentation seiner Bedeutung.

Wenn unser Vokabular **87 Tokens** enthält und wir eine Embedding-Größe von **96** wählen, hat die Embedding-Tabelle die Form:

* **87 Zeilen** → eine Zeile für jedes Token im Vokabular
* **96 Spalten** → jedes Token wird durch **96 Zahlen** beschrieben

Die Größe **96** legen wir selbst fest. Welche konkreten Werte darin stehen, **lernt das Modell während des Trainings**.

Werden mehrere Tokens zu einem Wort oder Satz kombiniert, summiert das Modell ihre Embeddings gemeinsam. Dadurch entsteht eine Repräsentation für die **Bedeutung im jeweiligen Kontext**.

Das Spannende daran: **Ähnliche Bedeutungen liegen im Vektorraum näher beieinander.**

Zum Beispiel könnten die Repräsentationen von **„Queen“** und **„Frau“** näher beieinanderliegen, während **„Queen“** und **„Auto“** deutlich weiter voneinander entfernt sind.

In [ ]:
# ▶️ Die Einstellungen, die wir hier brauchen
N_EMBD = 96       # Länge der Vektoren, mit denen das Modell intern arbeitet
KONTEXT = 96      # wie viele Tokens das Modell höchstens gleichzeitig sieht

token_tabelle = torch.randn(vokabular_groesse, N_EMBD)   # zufällig, vor dem Training

anzahl = f"{vokabular_groesse * N_EMBD:,}".replace(",", ".")
print(f"Token-Tabelle: {tuple(token_tabelle.shape)}  = {anzahl} Parameter")

### 🛠️ Challenge 3 — Vom One-Hot-Vektor zum Embedding

Wir haben bereits gesehen:

Ein Token wie **`c`** kann als One-Hot-Vektor dargestellt werden:

`c → [0, 0, 1, 0, 0, ...]`

Die **1** bedeutet dabei einfach:

> „Nimm die dritte Zeile aus der Embedding-Tabelle.“

Genau das können wir auch mit einer **Matrixmultiplikation** machen.

Angenommen, unsere Embedding-Tabelle hat:

* **87 Zeilen** → eine für jedes Token
* **96 Spalten** → 96 Zahlen pro Embedding

Dann hat sie die Form:

`(87, 96)`

Für einen Satz mit **5 Tokens** erzeugen wir zunächst für jedes Token einen One-Hot-Vektor, dadurch ensteht eine Matrix mit der Form:

`(5, 87)`

Jetzt multiplizieren wir:

`(5, 87) @ (87, 96) → (5, 96)`

Das Ergebnis enthält für jedes der **5 Tokens** genau den passenden **96-dimensionalen Embedding-Vektor**.

### Deine Aufgabe

Schreibe die Funktion:

`schlage_nach(tabelle, ids)`

Sie soll:

1. die Token-IDs in One-Hot-Vektoren umwandeln,
2. die One-Hot-Matrix mit der Embedding-Tabelle multiplizieren,
3. die Embeddings zurückgeben.

Verwende dafür **nicht** `tabelle[ids]`, sondern bewusst die Matrixmultiplikation.

**Hilfreiche Befehle:**

* `F.one_hot(ids, num_classes=...)` → erzeugt die One-Hot-Vektoren
* `tabelle.shape[0]` → Anzahl der Tokens im Vokabular
* `.float()` → wandelt sie in Fließkommazahlen um, PyTorch rechnet nur mit Fließkommazahlen.
* `a @ b` → Matrixmultiplikation in PyTorch

In [ ]:
def schlage_nach(tabelle, ids):
    """(V, C)-Tabelle und (T,) Token-Nummern → (T, C) Vektoren."""
    # TODO 1: One-Hot-Matrix bauen, Form (T, V)
    one_hot = ...

    # TODO 2: mit der Tabelle multiplizieren, Form (T, C)
    return ...

In [ ]:
# ✅ Selbsttest
vektoren = schlage_nach(token_tabelle, probe_ids)
assert vektoren.shape == (5, N_EMBD), f"Erwartet (5, {N_EMBD}), bekommen {tuple(vektoren.shape)}"
assert torch.allclose(vektoren, token_tabelle[probe_ids], atol=1e-5), \
    "Das Ergebnis muss Zeile für Zeile die Tabelle sein"
assert torch.allclose(schlage_nach(token_tabelle, torch.tensor([3])), token_tabelle[3:4], atol=1e-5)
print("✅ Challenge 3 gelöst")
print()
print(f"Eingabe:  {probe_ids.tolist()}   ← 5 Zeichen als Zahlen")
print(f"Ausgabe:  {tuple(vektoren.shape)}   ← 5 Vektoren mit je {N_EMBD} Zahlen")
print()
print("Der Vektor für 'F', erste 8 von 96 Zahlen:")
print(vektoren[0, :8].numpy().round(3))

<details>
<summary>💡 Lösung aufklappen — erst selbst probieren!</summary>

```python
def schlage_nach(tabelle, ids):
    """(V, C)-Tabelle und (T,) Token-Nummern → (T, C) Vektoren."""
    one_hot = F.one_hot(ids, num_classes=tabelle.shape[0]).float()

    return one_hot @ tabelle
```

</details>

In [ ]:
# ▶️ Genau das macht nn.Embedding — nur ohne die große One-Hot-Matrix dazwischen
embedding = nn.Embedding(vokabular_groesse, N_EMBD)
embedding.weight.data.copy_(token_tabelle)

print("Gleiches Ergebnis wie deine Funktion?",
      torch.allclose(embedding(probe_ids), schlage_nach(token_tabelle, probe_ids), atol=1e-5))
print()
print(f"Unser Token-Embedding:  {vokabular_groesse:>6,} × {N_EMBD:>3} = "
      f"{vokabular_groesse * N_EMBD:>12,} Parameter".replace(",", "."))
print(f"GPT-2 (klein):          {50257:>6,} × {768:>3} = {50257 * 768:>12,} Parameter"
      .replace(",", "."))

📖 38,6 Millionen Parameter allein für das Nachschlagen, das ist knapp ein Drittel des ganzen
GPT-2 (124 Millionen). Bei größeren Modellen fällt der Anteil, weil das Vokabular nicht
mitwächst: GPT-3 hat 617 Millionen von 175 Milliarden im Token-Embedding.

▶️ Die nächste Zelle zählt für die häufigsten Wörter im Faust, welche anderen Wörter in ihrer
Nähe stehen, und fasst diese Statistik zu Vektoren zusammen. Das ist nicht unser Modell, sondern das
klassische Verfahren aus der Zeit vor word2vec — aber das Signal ist dasselbe.

In [ ]:
# ▶️ Vektoren allein aus Nachbarschaft
woerter = re.findall(r"[a-zäöüß]+", text.lower())
haeufig = [w for w, n in Counter(woerter).most_common() if n >= 15]
platz = {w: i for i, w in enumerate(haeufig)}
folge = np.array([platz[w] for w in woerter if w in platz])

# Wie oft stehen zwei Wörter höchstens vier Plätze voneinander entfernt?
nachbarn = np.zeros((len(haeufig), len(haeufig)))
for abstand in (1, 2, 3, 4):
    np.add.at(nachbarn, (folge[:-abstand], folge[abstand:]), 1.0)
    np.add.at(nachbarn, (folge[abstand:], folge[:-abstand]), 1.0)

# Häufigkeit herausrechnen und die Statistik auf 32 Zahlen je Wort reduzieren
p = nachbarn / nachbarn.sum()
gewicht = np.maximum(np.log((p + 1e-12) / (p.sum(1, keepdims=True) * p.sum(0, keepdims=True) + 1e-12)), 0)
u, s, _ = np.linalg.svd(gewicht)
wort_vektoren = u[:, 1:33] * s[1:33]
wort_vektoren /= np.linalg.norm(wort_vektoren, axis=1, keepdims=True)

print(f"{len(haeufig)} Wörter, je {wort_vektoren.shape[1]} Zahlen")
print()
for wort in ["faust", "sonne", "herz", "der"]:
    aehnlich = np.argsort(-(wort_vektoren @ wort_vektoren[platz[wort]]))[1:6]
    print(f"  am ähnlichsten zu {wort!r}:  " + ", ".join(haeufig[i] for i in aehnlich))

In [ ]:
# ▶️ Dieselben Vektoren als Ähnlichkeitsmatrix
auswahl = ["faust", "mephistopheles", "margarete", "marthe",
           "der", "die", "den", "des",
           "himmel", "erde", "feuer", "wasser"]
teil = np.array([wort_vektoren[platz[w]] for w in auswahl])
aehnlichkeit = teil @ teil.T

plt.figure(figsize=(7, 6))
plt.imshow(aehnlichkeit, cmap="RdBu_r", vmin=-1, vmax=1)
plt.xticks(range(len(auswahl)), auswahl, rotation=90)
plt.yticks(range(len(auswahl)), auswahl)
plt.colorbar(label="Ähnlichkeit")
plt.title("Personen, Artikel, Elemente — je drei Blöcke")
plt.grid(False)
plt.show()

📖 Drei Blöcke auf der Diagonale, und niemand hat sie hineingeschrieben. Die Personen des Stücks
stehen beieinander, die Artikel stehen beieinander, die vier Elemente stehen beieinander — allein
daraus, welche Wörter im Faust nebeneinander vorkommen.

Genau dieses Signal nutzt das Training des Token-Embeddings, nur indirekt: Die Tabelle wird so
verändert, dass die Vorhersage des nächsten Tokens besser wird — und dabei landen Tokens mit
ähnlicher Rolle nebeneinander.

💬 **Wir haben die Vektorlänge auf 96 gesetzt. Was passiert bei 2 — und was bei 100.000?**

<details>
<summary>Antwort aufklappen</summary>

Bei **2** passt zu wenig hinein. Jedes Token bekäme zwei Zahlen; ähnliche Tokens müssten sich
denselben Platz teilen, und das Modell könnte sie nicht mehr auseinanderhalten. Die Vektorlänge
begrenzt, wie viele verschiedene Eigenschaften ein Token gleichzeitig tragen kann.

Bei **100.000** wächst der Rechenaufwand in jeder Schicht mit, ohne dass mehr Information da
wäre — die kommt aus den Daten, nicht aus der Tabelle. Zu große Embeddings sind vor allem teuer,
und das Modell neigt eher dazu, Trainingsdaten auswendig zu lernen.

Übliche Größen: 768 bei GPT-2 (klein), 4.096 bei Llama 2 (70 B). Es ist eine Einstellung wie die
Zahl der Schichten — man wählt sie im Verhältnis zu Datenmenge und Rechenbudget.

</details>

---
## 3 · Positional Encoding: wo steht das Token?

📖 Jetzt fehlt noch eine Information, die im Flussdiagramm als eigener Kasten neben dem Embedding
steht. Der Grund dafür ist der Baustein, der als Nächstes kommt: Die **Aufmerksamkeit** behandelt
die Eingabe als **Menge**, nicht als Reihenfolge. Für sie sind

> Der Hund biss den Mann.

und

> Der Mann biss den Hund.

**rechnerisch identisch** — dieselben Tokens, nur anders sortiert.

▶️ Man sieht es an zwei gleichen Zeichen an verschiedenen Stellen:

In [ ]:
# ▶️ Zwei Mal dasselbe Zeichen, zwei Mal derselbe Vektor
satz = "Der Hund biss den Mann."
ids = torch.tensor(kodiere(satz))
tok = schlage_nach(token_tabelle, ids)

a = satz.index("Hund") + 2      # das 'n' in "Hund"
b = satz.index("Mann") + 2      # das 'n' in "Mann"

print(f"Stelle {a}: {satz[a]!r}     Stelle {b}: {satz[b]!r}")
print(f"Beide Vektoren identisch?  {torch.allclose(tok[a], tok[b])}")

📖 Die Lösung ist schlicht: eine **zweite Tabelle**, diesmal nicht für Tokens, sondern für
Positionen — eine Zeile für Stelle 0, eine für Stelle 1, und so weiter bis 95. Beide Vektoren
werden **addiert**:

$$x = \text{Token-Embedding}(\text{Token}) + \text{Positions-Embedding}(\text{Stelle})$$

Damit trägt jeder Vektor beides: *was* dort steht und *wo* es steht. Das ist der Kasten `Add` im
Flussdiagramm.

### 🛠️ Challenge 4 — Token und Position zusammenbringen

`vorbereiten(ids)` bekommt die Token-Nummern eines Textes und gibt die Vektoren zurück, die in
den ersten Block gehen — Form `(T, 96)`.

Drei Schritte:

1. die Token-Vektoren nachschlagen — mit deinem `schlage_nach` und `token_tabelle`
2. die Positions-Vektoren nachschlagen — dieselbe Funktion, aber mit `positions_tabelle` und den
   Nummern `0, 1, 2, …, T-1`
3. beides addieren

*Tipp: Die Nummern `0` bis `T-1` liefert `torch.arange(T)`.*

In [ ]:
positions_tabelle = torch.randn(KONTEXT, N_EMBD)    # eine Zeile je Stelle im Kontext


def vorbereiten(ids):
    """(T,) Token-Nummern → (T, N_EMBD) Vektoren mit Inhalt und Position."""
    T = len(ids)

    # TODO 1: die Token-Vektoren
    tok = ...

    # TODO 2: die Positions-Vektoren für die Stellen 0 bis T-1
    pos = ...

    # TODO 3: beides zusammen
    return ...

In [ ]:
# ✅ Selbsttest
x = vorbereiten(ids)
assert x.shape == (len(ids), N_EMBD), f"Erwartet {(len(ids), N_EMBD)}, bekommen {tuple(x.shape)}"
assert not torch.allclose(x[a], x[b]), \
    "Gleiches Zeichen an verschiedenen Stellen muss jetzt verschiedene Vektoren geben"
assert torch.allclose(x[0], token_tabelle[ids[0]] + positions_tabelle[0], atol=1e-5), \
    "An Stelle 0 muss die Summe aus Token-Vektor und Positions-Vektor 0 stehen"
kurz = vorbereiten(torch.tensor(kodiere("Der Hund")))
assert torch.allclose(kurz[3], x[3], atol=1e-5), \
    "Dieselbe Stelle im selben Textanfang muss denselben Vektor geben"
print("✅ Challenge 4 gelöst")
print()
print(f"Token-Vektoren:      {tuple(schlage_nach(token_tabelle, ids).shape)}")
print(f"Positions-Vektoren:  {tuple(schlage_nach(positions_tabelle, torch.arange(len(ids))).shape)}")
print(f"Summe:               {tuple(x.shape)}   ← das geht in den ersten Block")
print()
print(f"Stelle {a} und {b} ({satz[a]!r} und {satz[b]!r}) jetzt noch identisch?  "
      f"{torch.allclose(x[a], x[b])}")

<details>
<summary>💡 Lösung aufklappen — erst selbst probieren!</summary>

```python
positions_tabelle = torch.randn(KONTEXT, N_EMBD)    # eine Zeile je Stelle im Kontext


def vorbereiten(ids):
    """(T,) Token-Nummern → (T, N_EMBD) Vektoren mit Inhalt und Position."""
    T = len(ids)

    tok = schlage_nach(token_tabelle, ids)

    pos = schlage_nach(positions_tabelle, torch.arange(T))

    return tok + pos
```

Dass hier addiert und nicht aneinandergehängt wird, überrascht oft. Aneinanderhängen würde die
Vektoren doppelt so lang machen und damit jede folgende Schicht verteuern. Beim Addieren bleibt
die Länge gleich — das Modell hat 96 Dimensionen zur Verfügung und lernt selbst, welche davon es
für Inhalt und welche es für Position benutzt.

</details>

---
## 4 · Alles zusammen

📖 Damit ist der Eingang fertig. Aus Text wird eine Zahlenfolge, aus jeder Zahl ein Vektor, dazu
kommt die Position — und heraus kommt eine Matrix der Form `(Batch, Länge, 96)`.

Die dritte Dimension ist die, mit der der Transformer von hier an arbeitet. Die erste, der
**Batch**, ist nur die Zahl der Textstücke, die gleichzeitig durchgerechnet werden; im Training
sind das viele, hier eines.

In [ ]:
# ▶️ Der komplette Eingang, in einer Funktion
def eingabe_vorbereiten(s):
    """Text → (1, T, N_EMBD): der Tensor, der in den ersten Transformer-Block geht."""
    ids = torch.tensor(kodiere(s[-KONTEXT:]))       # länger als der Kontext geht nicht
    return vorbereiten(ids).unsqueeze(0)            # Batch-Dimension davor


x = eingabe_vorbereiten("Habe nun, ach! Philosophie")

print("Habe nun, ach! Philosophie")
print(f"  Tokenizer          → {len(kodiere('Habe nun, ach! Philosophie'))} Token-Nummern")
print(f"  Word Embedding     → {len(kodiere('Habe nun, ach! Philosophie'))} × {N_EMBD} Zahlen")
print(f"  + Positional       → {tuple(x.shape)}   (Batch, Länge, Dimension)")
print()
print("Die ersten 6 Zahlen des ersten Vektors:")
print(x[0, 0, :6].numpy().round(3))

📖 Zwei Zahlen zum Mitnehmen: Unser Eingang hat 8.352 + 9.216 = **17.568 Parameter** — beides
Tabellen, in denen nachgeschlagen wird. Bei GPT-2 sind es 38.597.376 + 786.432.

---
### 🔬 Zum Weiterprobieren

Alles hier darf geändert werden — die Zellen darunter brauchen nichts davon.

In [ ]:
# ▶️ Spielfeld
mein_text = "Mephistopheles"

print(f"Zeichen-Tokenizer:  {len(kodiere(mein_text))} Tokens")
if GPT2 is not None:
    print(f"GPT-2-Tokenizer:    {len(GPT2.encode(mein_text))} Tokens  {gpt2_stuecke(mein_text)}")
print(f"Vorbereitet:        {tuple(eingabe_vorbereiten(mein_text).shape)}")

**Drei Ideen:**

1. **Tokenizer vergleichen.** Schicke deinen eigenen Namen, eine Zahl wie `1234567`, eine
   E-Mail-Adresse und ein englisches Wort durch `gpt2_stuecke()`. Wo zerfällt der Text in viele
   Stücke, wo bleibt er ganz?

2. **Mehr Verschmelzungen.** Setze `VERSCHMELZUNGEN` auf 500 und führe die BPE-Zelle noch einmal
   aus. Welche Wortteile kommen dazu? Wie viele Zeichen je Token sind es dann?

3. **Kontextlänge.** Schicke einen Text mit mehr als 96 Zeichen durch `eingabe_vorbereiten()`.
   `KONTEXT` begrenzt die Länge — welcher Teil des Textes bleibt übrig, und was steht danach in
   der ersten Zeile des Ergebnisses?